# 01 · Ticket routing: your first decision model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daseinlabs/open-jev/blob/main/cookbooks/01-ticket-routing/cookbook.ipynb)


Turn a CSV of labelled support tickets into a model that picks a **team** and a **priority** for
new tickets, says how confident it is, and hands off the tickets it isn't sure about.

You'll touch every part of the SDK once: a `DecisionSpec`, a dataset, training, calibration, and
confident-or-defer serving. About 10–20 minutes, mostly Gemma reading the tickets.

```
tickets.csv ──spec──▶ decision dataset ──train──▶ calibrated model ──decide──▶ team / priority / defer
```


**Two methods**, chosen with `METHOD` in the setup cell; everything else is the same:

- `'tod'` (default) fine-tunes [TOD](https://huggingface.co/parsecai/tod) (Gemma 4 12B + LoRA). Needs an
  NVIDIA GPU with 40 GB+ (A100, H100, L40S; on Colab an A100 runtime) or a 64 GB Apple silicon Mac.
  Downloads `google/gemma-4-12B-it` (24 GB, not gated) once.
- `'head'` trains small heads on a frozen Gemma 3 4B. Needs ~9 GB (any recent NVIDIA GPU, or any Apple
  silicon Mac). Downloads [`parsecai/gemma-3-4b-it`](https://huggingface.co/parsecai/gemma-3-4b-it)
  (8 GB, gated: accept the Gemma terms there first).

In [ ]:
METHOD = 'tod'  # 'tod': fine-tune TOD (40 GB+ GPU) | 'head': small heads on Gemma 3 4B (~9 GB)

# Setup: install the SDK if it's missing, log in to Hugging Face once (for the gated Gemma 3 backbone).
import importlib.util
if importlib.util.find_spec('openjev') is None:
    %pip install -q openjev-sdk

from huggingface_hub import get_token, login
if METHOD == 'head' and get_token() is None:
    login()

import json, shutil
from pathlib import Path
from openjev import compute
from openjev.tod.picker import finetune, release
from openjev.tod.rl import load_policy
from openjev.tod.train import train_dataset

def train(dataset, out):
    # fine-tune TOD, or train heads; then calibrate and report test accuracy per kind
    return finetune(dataset, out) if METHOD == 'tod' else train_dataset(dataset, None, out)

def fresh(path):
    """Remove a previous run's output, so every cell can be re-run."""
    shutil.rmtree(path, ignore_errors=True)
    return Path(path)

print('method:', METHOD, '| device:', compute.device() if METHOD == 'tod' or compute.backend() == 'torch' else 'metal (MLX)')

## 0. The data

A sample of 1,200 labelled tickets. 30% mention two topics, and 5% are deliberately mislabelled,
as real routing logs are. **For your own data, export the same columns and skip this cell.**

In [ ]:
import csv, random

TEMPLATES = {
    'billing': ["I was charged twice for {thing}.", "My invoice for {thing} shows the wrong amount.",
                "Can I get a refund on the {thing} subscription fee?", "Why did my card get billed again this month?"],
    'shipping': ["My {thing} still hasn't arrived.", "The tracking for my {thing} hasn't moved in a week.",
                 "Can you ship the {thing} to a different address?", "The courier left my {thing} at the wrong house."],
    'returns': ["The {thing} arrived broken, I want a replacement.", "How do I send back the {thing}?",
                "The {thing} is the wrong size, can I exchange it?", "I'd like to return the {thing} I bought last week."],
    'account': ["I can't log in to my account.", "Please change the email on my account.",
                "I want to delete my account and data.", "My password reset link isn't working."],
    'technical': ["The app crashes when I open my {thing} order.", "The website shows an error at checkout.",
                  "Your app won't load my order history.", "The {thing} setup guide link is broken."],
}
THINGS = ['headphones', 'blender', 'yoga mat', 'monitor', 'jacket', 'coffee maker', 'backpack', 'desk']
URGENT = ["This is urgent.", "I need this fixed today.", "Please help ASAP!", "This is the third time I'm asking."]
LOW = ["No rush.", "Whenever you get a chance.", "Just curious.", "Not a big deal."]

rng = random.Random(0)
with open('tickets.csv', 'w', newline='') as stream:
    writer = csv.writer(stream)
    writer.writerow(['ticket_id', 'customer_id', 'message', 'team', 'priority'])
    for i in range(1200):
        team = rng.choice(list(TEMPLATES))
        priority = rng.choices(['urgent', 'normal', 'low'], [1, 2, 1])[0]
        message = rng.choice(TEMPLATES[team]).format(thing=rng.choice(THINGS))
        if rng.random() < .3:  # ambiguous tickets mention a second topic
            other = rng.choice([t for t in TEMPLATES if t != team])
            message += ' Also, ' + rng.choice(TEMPLATES[other]).format(thing=rng.choice(THINGS)).lower()
        if rng.random() < .6:
            message += f' Order #{rng.randint(10000, 99999)}.'
        if priority != 'normal':
            message += ' ' + rng.choice(URGENT if priority == 'urgent' else LOW)
        if rng.random() < .05:  # real labels are noisy: some tickets were routed by mistake
            team = rng.choice(list(TEMPLATES))
        writer.writerow([f't{i:05d}', f'c{rng.randrange(400):04d}', message, team, priority])

tickets = list(csv.DictReader(open('tickets.csv')))
for t in tickets[:3]:
    print(t)

## 1. Describe the decisions

A **kind** is one type of decision: a question, plus a function that lists the candidates for a
state. Here both candidate lists are fixed.

- Candidate **text** is what the model reads, so describe each option
  (`'Returns: broken items, exchanges, send-backs'`, not just `returns`).
- The **payload** is what your code gets back.
- Bump a kind's `version` whenever its candidate list changes. The SDK then refuses to serve an
  old model with the new spec.

In [ ]:
from openjev.tod import Candidate, DecisionSpec, Episode, Kind, build_dataset

TEAMS = {'billing': 'Billing: charges, invoices, refunds of fees',
         'shipping': 'Shipping: delivery, tracking, addresses',
         'returns': 'Returns: broken items, exchanges, send-backs',
         'account': 'Account: login, email, deletion',
         'technical': 'Technical: app or website errors'}
PRIORITIES = {'urgent': 'Urgent: handle today', 'normal': 'Normal', 'low': 'Low: no rush'}

def options(table):
    return lambda state: [Candidate(id, text, payload=id) for id, text in table.items()]

SPEC = DecisionSpec('ticket-routing', [
    Kind('team', 'Which team should handle this ticket?', options(TEAMS), version='1'),
    Kind('priority', 'How urgent is this ticket?', options(PRIORITIES), version='1'),
])

## 2. Build the dataset

Every row goes through the spec, so a model can later check that its data and its serving code
agree. The **state** is any JSON; everything in it is shown to the model. `group_id` keeps all
of a customer's tickets in one split, so test accuracy isn't inflated by near-duplicates the model saw in training.

The report shows rows per split and kind. Identical tickets are kept once (`duplicates_dropped`),
so the same text can't sit in both train and test.

In [ ]:
episodes = []
for row in tickets:
    state = {'message': row['message']}
    episodes.append(Episode(row['ticket_id'], (
        SPEC.decision(f"{row['ticket_id']}:team", 'team', state, choice=row['team'], group_id=row['customer_id']),
        SPEC.decision(f"{row['ticket_id']}:priority", 'priority', state, choice=row['priority'], group_id=row['customer_id']),
    )))

dataset = build_dataset(SPEC, episodes)
DATASET = fresh('runs/tickets')
dataset.write(DATASET)
print(json.dumps({k: dataset.report[k] for k in ('splits', 'duplicates_dropped', 'duplicates_with_different_label')}, indent=1))

## 3. Train

- **`tod`:** TOD's adapter keeps training on your tickets with cross-entropy over its answer letters,
  for 2 epochs; the checkpoint with the best validation accuracy is kept. The report also shows
  zero-shot TOD on the same test split. Terminal: `openjev tod train runs/tickets --method tod --out runs/tickets-model`.
- **`head`:** each kind gets a small head (~2M parameters) on the frozen backbone; features are
  cached first. Terminal: `openjev tod train runs/tickets --out runs/tickets-model`.

Both then fit a temperature per kind on the calibration split, so "0.9 confident" means right
about 90% of the time, and report accuracy and mean confidence on test tickets never used above.

In [ ]:
MODEL = fresh('runs/tickets-model')
result = train(DATASET, MODEL)

In [ ]:
for kind, r in result['test'].items():
    print(f"{kind:9} accuracy {r['accuracy']:.1%}   mean confidence {r['mean_confidence']:.2f}   n={r['n']}")

Team routing is the harder kind: some tickets mention two topics, and 5% of labels are wrong
(test included). Mean confidence close to accuracy means the probabilities are honest.

## 4. Serve: act when confident, defer otherwise

In [ ]:
policy = load_policy(MODEL, SPEC)  # refuses a model trained with a different spec

for message in ['My parcel never showed up and I need it for tomorrow!',
                'I was charged twice and also my parcel is late',
                'hello?']:
    print(message)
    for kind in SPEC.kinds:
        p = policy.decide({'message': message}, kind, threshold=0.8)
        print(f'  {kind:9} -> {p.choice_id:10} confidence {p.confidence:.2f}' + ('   DEFER' if p.defer else ''))

In your code:

```python
p = policy.decide(state, 'team', threshold=0.8)
if p.defer:
    ...                 # hand the ticket to a person or a bigger model
else:
    route_to(p.payload)
```

## 5. Pick the threshold

The threshold is the product dial: raise it, and the model makes fewer mistakes but hands off more.
Here is the trade-off on the test tickets:

In [ ]:
from openjev.tod.spec import load_decisions

test = [d for d in load_decisions(DATASET/'test.jsonl') if d.kind == 'team']
predictions = [(policy.decide(d.state, 'team'), d.target.choice_id) for d in test]
print('threshold  handled  accuracy on handled')
for threshold in (0.0, 0.5, 0.7, 0.8, 0.9, 0.95):
    kept = [(p, y) for p, y in predictions if p.confidence >= threshold]
    accuracy = sum(p.choice_id == y for p, y in kept) / max(1, len(kept))
    print(f'{threshold:9.2f}  {len(kept) / len(predictions):7.0%}  {accuracy:19.1%}')

## Your data

- **Columns:** change the loop in step 2. The state can hold anything (subject, body, customer tier, ...).
- **Candidates:** edit `TEAMS` / `PRIORITIES`, and bump that kind's `version`.
- **More kinds:** add a `Kind` and one more `SPEC.decision(...)` per record.

Next: [02 · traces → SFT → RL](../02-shopdesk-traces-to-rl/cookbook.ipynb).